# Graph attention on an edge list

The lesson's attention head builds an $n \times n$ score matrix and masks it, which works for a toy
graph and not for one with a million nodes. Graph libraries compute the same head from the edge
list instead: one score per edge, and a softmax taken separately over the edges that end at each
node. You write that softmax and the head in NumPy, for seven accounts on a site where following
is one-way.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

def leaky_relu(x, slope=0.2):
    return np.where(x > 0, x, slope * x)

# Seven accounts. An edge i -> j means account j sees the posts of account i, so j receives
# a message from i. Following is one-way, and nobody follows themselves.
edge_index = np.array([[0, 0, 1, 2, 3, 3, 4, 5, 6, 6],    # sources
                       [1, 2, 2, 0, 2, 4, 5, 3, 3, 5]])   # targets
n = 7
rng = np.random.default_rng(3)
H = rng.normal(size=(n, 4))     # account features
W = rng.normal(size=(4, 3))
a = rng.normal(size=6)          # attention vector: receiver half first, sender half second
print("incoming edges per account:", np.bincount(edge_index[1], minlength=n))

## Part 1 (coding task): A softmax within each group

Each node's attention weights are a softmax over the edges that end at it. With one score per
edge, that is a softmax taken separately within groups of edges. Write
`segment_softmax(scores, segment, n)`. `scores` holds one number per edge, and `segment[e]` is
the group of edge `e`, a number from 0 to $n - 1$. The function returns one weight per edge:
within each group, the softmax of that group's scores. A group may be empty.

Attention scores are not bounded, so the function has to work when they run into the hundreds
or thousands, positive or negative.

In [ ]:
def segment_softmax(scores, segment, n):
    """The softmax of scores taken within each group, where segment[e] is the group of entry e."""
    # TODO
    raise NotImplementedError

In [ ]:
small = segment_softmax(np.array([1.0, 2.0, 3.0, 0.5]), np.array([0, 0, 2, 0]), 3)
assert np.shape(small) == (4,), f"expected one weight per entry, shape (4,), got {np.shape(small)}"
e = np.exp([1.0, 2.0, 0.5])
assert np.isclose(small[2], 1), "an entry alone in its group should get weight 1"
assert np.allclose(small[[0, 1, 3]], e / e.sum()), (
    f"group 0 holds the scores 1, 2 and 0.5, so its weights should be {np.round(e / e.sum(), 4)}, "
    f"got {np.round(small[[0, 1, 3]], 4)}")

# Random scores in random groups. The weights of each group add up to 1, and inside a group a
# softmax fixes every ratio, w_e / w_f = exp(s_e - s_f), so log(w) - s is the same for the group.
r = np.random.default_rng(0)
seg = r.integers(0, 6, size=40)
s = r.normal(scale=3, size=40)
w = segment_softmax(s, seg, 8)
sums = np.bincount(seg, weights=w, minlength=8)
assert np.allclose(sums[np.bincount(seg, minlength=8) > 0], 1), (
    f"the weights of each group should add up to 1, got group sums {np.round(sums, 4)}")
for g in np.unique(seg):
    c = np.log(w[seg == g]) - s[seg == g]
    assert np.allclose(c, c[0]), f"the weights of group {g} are not a softmax of its scores"

big = segment_softmax(np.array([1000.0, 0.0, -1000.0, -999.0]), np.array([0, 0, 1, 1]), 2)
assert np.all(np.isfinite(big)), (
    f"scores of 1000 and -1000 give {big}: exp overflows above about 709, and a group whose terms "
    "all underflow to 0 divides 0 by 0")
t = 1 / (1 + np.e)
assert np.allclose(big, [1, 0, t, 1 - t]), f"expected [1, 0, {t:.4f}, {1 - t:.4f}], got {np.round(big, 4)}"
print("Part 1 passed")

## Part 2 (coding task): The attention head from the edge list

Write `gat_edge_list(edge_index, H, W, a, slope=0.2)`, the lesson's attention head computed from
the edge list, without any $n \times n$ array. Every node also attends to itself, and the edge
lists here have no self-loops, so first add the edge $v \to v$ for every node. With
$Z = HW$ of width $k$, an edge from $j$ to $i$ gets the score

$$
e_{ij} = \operatorname{LeakyReLU}\big(\mathbf{a}^\top[\mathbf{z}_i \,\Vert\, \mathbf{z}_j]\big),
$$

where $i$ is the receiver, so the first $k$ entries of `a` multiply the receiver's $\mathbf{z}$.
The weights $\alpha_{ij}$ are the softmax of the scores over the edges that end at $i$, and node
$i$'s output is $\sum_j \alpha_{ij} \mathbf{z}_j$ over those edges. Return the $n \times k$ array
of outputs.

In [ ]:
def gat_edge_list(edge_index, H, W, a, slope=0.2):
    """One attention head from a 2 x E edge list (sources, then targets), self-loops added."""
    # TODO
    raise NotImplementedError

In [ ]:
def dense_head(H, A_hat, W, a, slope=0.2):
    """The lesson's head: node i attends to node j when A_hat[i, j] > 0."""
    Z = H @ W
    k = Z.shape[1]
    e = leaky_relu((Z @ a[:k])[:, None] + (Z @ a[k:])[None, :], slope)
    e = np.where(A_hat > 0, e, -np.inf)
    alpha = np.exp(e - e.max(axis=1, keepdims=True))
    return (alpha / alpha.sum(axis=1, keepdims=True)) @ Z

def flip(a):
    k = len(a) // 2
    return np.concatenate([a[k:], a[:k]])

src, dst = edge_index
A_hat = np.eye(n)
A_hat[dst, src] = 1            # row = receiver, column = sender, plus self-loops
out = gat_edge_list(edge_index, H, W, a)
assert np.shape(out) == (n, 3), f"expected an output of shape ({n}, 3), got {np.shape(out)}"
expected = dense_head(H, A_hat, W, a)
if not np.allclose(out, expected):
    if np.allclose(out, dense_head(H, A_hat, W, flip(a))):
        raise AssertionError("the two halves of a are swapped: the first k entries go with the receiver, edge_index[1]")
    if np.allclose(out, dense_head(H, A_hat.T, W, a)) or np.allclose(out, dense_head(H, A_hat.T, W, flip(a))):
        raise AssertionError("every node attends over the edges that leave it: the receiver of an edge is its target, edge_index[1]")
    if not np.allclose(out[6], (H @ W)[6]):
        raise AssertionError(
            "account 6 follows nobody, so no edge ends at it and it attends only to itself: its output "
            "should be its own z. Does every node have a self-loop, and do each node's weights add up to 1?")
    raise AssertionError(
        "account 6, whose only edge is its self-loop, comes out right, and some accounts with incoming "
        "edges do not: look at how the messages of several edges are combined at one receiver")
assert np.allclose(gat_edge_list(edge_index, H, W, a, slope=0.5), dense_head(H, A_hat, W, a, slope=0.5)), (
    "the slope argument is not used")

# A larger random graph with no self-loops and no repeated edges.
r = np.random.default_rng(1)
pairs = [(i, j) for i in range(30) for j in range(30) if i != j]
chosen = np.array([pairs[t] for t in r.choice(len(pairs), size=120, replace=False)]).T
H2, W2, a2 = r.normal(size=(30, 5)), r.normal(size=(5, 4)), r.normal(size=8)
A2 = np.eye(30)
A2[chosen[1], chosen[0]] = 1
assert np.allclose(gat_edge_list(chosen, H2, W2, a2), dense_head(H2, A2, W2, a2)), (
    "right on the seven accounts but wrong on a random graph of 30 nodes")
print("Part 2 passed")

## Solution

### Part 1: A softmax within each group

In [ ]:
def segment_softmax(scores, segment, n):
    """The softmax of scores taken within each group, where segment[e] is the group of entry e."""
    top = np.full(n, -np.inf)
    np.maximum.at(top, segment, scores)       # the largest score of each group
    e = np.exp(scores - top[segment])         # at most 1, and exactly 1 at each group's top
    total = np.zeros(n)
    np.add.at(total, segment, e)
    return e / total[segment]

**Why it works.** A softmax does not change when every score of a group moves by the same amount, since the
factor $e^{-m}$ cancels between the numerator and the group's total. Taking $m$ as the group's
own largest score makes every exponent at most 0, so nothing overflows, and the top entry of each
group contributes exactly 1, so no total is 0. A single shift for all groups fails: in the last
check the largest score is 1000, and shifting the second group by it leaves exponents near $-2000$,
which underflow to 0 and give $0/0$. `np.maximum.at` and `np.add.at` are needed because a group
appears many times in `segment`. A plain `top[segment] = np.maximum(top[segment], scores)` keeps
only the last write to each group, which here is the 0 in group 0, and $e^{1000}$ overflows.

### Part 2: The attention head from the edge list

In [ ]:
def gat_edge_list(edge_index, H, W, a, slope=0.2):
    """One attention head from a 2 x E edge list (sources, then targets), self-loops added."""
    n = H.shape[0]
    loops = np.arange(n)
    src = np.concatenate([edge_index[0], loops])
    dst = np.concatenate([edge_index[1], loops])
    Z = H @ W
    k = Z.shape[1]
    e = leaky_relu(Z[dst] @ a[:k] + Z[src] @ a[k:], slope)   # one score per edge, receiver first
    alpha = segment_softmax(e, dst, n)                        # normalized over each receiver's edges
    out = np.zeros_like(Z)
    np.add.at(out, dst, alpha[:, None] * Z[src])              # every receiver sums its weighted messages
    return out

**Why it works.** Concatenating the self-loops to both rows of the edge list puts them through the same code as
every other edge, so account 6, which receives nothing, attends to itself alone and keeps its
own $\mathbf{z}$. Indexing `Z[dst]` and `Z[src]` gathers the receiver's and the sender's vectors
for every edge at once, which turns $\mathbf{a}^\top[\mathbf{z}_i \Vert \mathbf{z}_j]$ into two
matrix-vector products. Grouping the softmax by `dst` makes it run over each receiver's
neighborhood, the same set the dense head keeps with its $-\infty$ mask, and `np.add.at` adds every
weighted message into its receiver even when several edges share one. Memory and time grow with
the number of edges, not with $n^2$. On this graph a mistake in direction cannot hide: following
is one-way, so the edges that end at a node are not the edges that leave it.